# Sistema de alerta temprana de sequia (Mallacayan) — inferencia del modelo embebido

Este notebook corre en Google Colab y hace **dos cosas reales**, sin simular nada:

1. **Adquisicion satelital/climatica real** contra Google Earth Engine (Sentinel-2, MODIS, CHIRPS, ERA5-Land) para el cuadrante de Mallacayan — el mismo codigo (adaptado) que corre en el pipeline de produccion `lima_cloud`.
2. **Inferencia real del Random Forest embebido**: el codigo de `votesForOnset()` de abajo es un **port literal** (si/entonces por si/entonces, mismos 40 arboles, mismos umbrales) del archivo `drought_onset_model.h` ya compilado en el ESP32-S3 — no es un modelo reentrenado ni una aproximacion.

**Lo unico que Colab no puede hacer es leer el sensor BME280 real** (no hay forma de conectar el hardware de campo a una VM de Colab), asi que esa parte se ingresa **manualmente** en la Sección 6, con la ultima lectura real que tengas del nodo.

---
**Honestidad de este notebook** (léase antes de usar los resultados):
- `dsi_score` y `cur_class` siguen siendo la heurística propia documentada en `lima_cloud` (no la fórmula exacta de entrenamiento — ver advertencia en la Sección 4).
- El sensor local **no entra como feature cruda del vector x[0..19]** — el firmware real tampoco lo hace: el vector son 16 variables satelitales + fila/columna del cuadrante + mes (seno/coseno). El sensor sirve como **validación de plausibilidad** (Sección 6), igual que `sensor_validator.cpp` en el firmware.
- El port de `votesForOnset()` se verificó comparando **dos transpilaciones independientes** del mismo `drought_onset_model.h` (un tokenizador por regex y un parser recursivo por conteo de llaves) sobre 20 000 vectores aleatorios: 0 discrepancias. No se pudo compilar el `.h` original en este entorno (sin `g++`) para un diff bit a bit contra el firmware real — si tenés `g++` a mano, es la verificación definitiva.

## 1. Instalación

In [ ]:
!pip install -q earthengine-api numpy scipy

## 2. Autenticación a Google Earth Engine

Corre esta celda y seguí el link: te va a pedir loguearte con la cuenta de Google que tiene acceso al proyecto `sequias-iac` (o el proyecto de GEE que uses) y pegar el código de autorización.

In [ ]:
import ee

ee.Authenticate()  # abre el flujo OAuth interactivo de Colab
GEE_PROJECT = "sequias-iac"  # cambiar si tu proyecto de Google Cloud/Earth Engine tiene otro id
ee.Initialize(project=GEE_PROJECT)
print("Earth Engine inicializado con el proyecto:", GEE_PROJECT)

## 3. Configuración: grilla, cuadrante y coordenadas reales de Mallacayán

Mismos valores que `lima_cloud/config.py` (coordenadas GPS reales del nodo: lat. −9.718924, lon. −77.598083; recuadro de ±0.075° alrededor; grilla 6×6; el nodo cae en el cuadrante `3_3`).

In [ ]:
BASIN_LAT_MIN, BASIN_LAT_MAX = -9.793924, -9.643924
BASIN_LON_MIN, BASIN_LON_MAX = -77.673083, -77.523083
GRID_ROWS, GRID_COLS = 6, 6
NODE_QUADRANT_ROW, NODE_QUADRANT_COL = 3, 3

S2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"
MODIS_LST_COLLECTION = "MODIS/061/MOD11A2"
CHIRPS_COLLECTION = "UCSB-CHG/CHIRPS/DAILY"
ERA5_LAND_COLLECTION = "ECMWF/ERA5_LAND/DAILY_AGGR"

S2_LOOKBACK_DAYS = 10
LST_LOOKBACK_DAYS = 35       # MOD11A2 tiene ~17 dias de latencia real de publicacion
ERA5_LOOKBACK_DAYS = 20
TREND_WINDOW_DAYS = 14
CDD_WINDOW_DAYS = 60
SPI_HISTORY_YEARS = 15       # CHIRPS: la gamma necesita historico largo, no la ventana reciente
VCI_HISTORY_YEARS = 6
CDD_DRY_THRESHOLD_MM = 1.0
VCI_CLASS_THRESHOLDS = (10.0, 20.0, 35.0, 50.0)

def build_grid():
    lat_step = (BASIN_LAT_MAX - BASIN_LAT_MIN) / GRID_ROWS
    lon_step = (BASIN_LON_MAX - BASIN_LON_MIN) / GRID_COLS
    quads = []
    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            quads.append({
                "row": r, "col": c, "id": f"{r}_{c}",
                "lat_min": BASIN_LAT_MIN + r * lat_step, "lat_max": BASIN_LAT_MIN + (r + 1) * lat_step,
                "lon_min": BASIN_LON_MIN + c * lon_step, "lon_max": BASIN_LON_MIN + (c + 1) * lon_step,
            })
    return quads

NODE_QUADRANT_ID = f"{NODE_QUADRANT_ROW}_{NODE_QUADRANT_COL}"
quad = next(q for q in build_grid() if q["id"] == NODE_QUADRANT_ID)
print("Cuadrante del nodo:", quad)

## 4. Adquisición satelital/climática real (Google Earth Engine)

Mismas funciones que `lima_cloud/indices.py`, adaptadas para correr sueltas en Colab (sin el módulo `config`). Cada función hace la reducción del lado del servidor en GEE y solo trae el número final a Python — el cómputo pesado (todas las bandas, todos los píxeles) queda en los servidores de Earth Engine, igual que en producción.

**Advertencia sobre `dsi_score`/`cur_class`**: son una propuesta heurística propia, documentada así en `lima_cloud` — no la fórmula exacta que usó el equipo de ML al entrenar `drought_onset_model.h`. Si no coinciden, el modelo recibe entradas fuera de su distribución de entrenamiento.

In [ ]:
import datetime as dt
import numpy as np
from scipy import stats


def quadrant_geometry(q):
    return ee.Geometry.Rectangle([q["lon_min"], q["lat_min"], q["lon_max"], q["lat_max"]])


def _mask_s2_clouds(img):
    scl = img.select("SCL")
    bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10))
    return img.updateMask(bad.Not())


def _reduce_mean(image, band, geom, scale):
    stat = image.select(band).reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=scale, maxPixels=1e9, bestEffort=True)
    val = stat.get(band).getInfo()
    return float(val) if val is not None else float("nan")


def _collection_is_empty(coll):
    return coll.size().getInfo() == 0


def ndvi_ndwi_now(geom, end_date, lookback_days):
    start = end_date.advance(-lookback_days, "day")
    raw = ee.ImageCollection(S2_COLLECTION).filterBounds(geom).filterDate(start, end_date).map(_mask_s2_clouds)
    if _collection_is_empty(raw):
        return float("nan"), float("nan")
    comp = raw.median()
    ndvi = comp.normalizedDifference(["B8", "B4"]).rename("ndvi")
    ndwi = comp.normalizedDifference(["B8", "B11"]).rename("ndwi")
    return _reduce_mean(ndvi, "ndvi", geom, 20), _reduce_mean(ndwi, "ndwi", geom, 20)


def lst_celsius(geom, end_date, lookback_days):
    start = end_date.advance(-lookback_days, "day")
    coll = ee.ImageCollection(MODIS_LST_COLLECTION).filterBounds(geom).filterDate(start, end_date)
    if _collection_is_empty(coll):
        return float("nan")
    img = coll.select("LST_Day_1km").mean().multiply(0.02).subtract(273.15).rename("lst")
    return _reduce_mean(img, "lst", geom, 1000)


def ndvi_series(geom, end_date, n_days):
    start = end_date.advance(-n_days, "day")
    coll = (ee.ImageCollection(S2_COLLECTION).filterBounds(geom).filterDate(start, end_date).map(_mask_s2_clouds)
            .map(lambda img: img.normalizedDifference(["B8", "B4"]).rename("ndvi").copyProperties(img, ["system:time_start"])))
    feats = coll.getRegion(geom.centroid(1), scale=20).getInfo()  # centroide: getRegion() sobre un rectangulo devuelve 0 filas
    if len(feats) <= 1:
        return np.array([])
    header, rows = feats[0], feats[1:]
    idx = header.index("ndvi")
    return np.array([r[idx] for r in rows if r[idx] is not None], dtype=float)


def linear_trend(series):
    if len(series) < 3:
        return float("nan")
    x = np.arange(len(series))
    slope, _ = np.polyfit(x, series, 1)
    return float(slope)


def precip_daily_series(geom, end_date, n_days):
    start = end_date.advance(-n_days, "day")
    coll = ee.ImageCollection(CHIRPS_COLLECTION).filterBounds(geom).filterDate(start, end_date)
    feats = coll.getRegion(geom.centroid(1), scale=5000).getInfo()
    if len(feats) <= 1:
        return np.array([])
    header, rows = feats[0], feats[1:]
    idx = header.index("precipitation")
    return np.array([r[idx] for r in rows if r[idx] is not None], dtype=float)


def era5_temps(geom, end_date):
    coll = (ee.ImageCollection(ERA5_LAND_COLLECTION).filterBounds(geom)
            .filterDate(end_date.advance(-ERA5_LOOKBACK_DAYS, "day"), end_date).sort("system:time_start", False))
    if _collection_is_empty(coll):
        nan = float("nan")
        return nan, nan, nan
    img = ee.Image(coll.first())
    tmax = _reduce_mean(img.select("temperature_2m_max"), "temperature_2m_max", geom, 10000) - 273.15
    tmin = _reduce_mean(img.select("temperature_2m_min"), "temperature_2m_min", geom, 10000) - 273.15
    tmean = (tmax + tmin) / 2.0
    return tmean, tmax, tmin


def ndvi_historical_minmax(geom, end_date, years):
    vals = []
    for y in range(1, years + 1):
        ref = end_date.advance(-y, "year")
        start, end = ref.advance(-15, "day"), ref.advance(15, "day")
        yearly = ee.ImageCollection(S2_COLLECTION).filterBounds(geom).filterDate(start, end).map(_mask_s2_clouds)
        if _collection_is_empty(yearly):
            continue
        ndvi = yearly.median().normalizedDifference(["B8", "B4"]).rename("ndvi")
        v = _reduce_mean(ndvi, "ndvi", geom, 20)
        if not np.isnan(v):
            vals.append(v)
    if len(vals) < 2:
        return float("nan"), float("nan")
    return float(np.min(vals)), float(np.max(vals))


print("Funciones de adquisición GEE listas.")

## 5. SPI, ET0 y variables derivadas

Cálculo local (no depende de GEE) sobre la serie de precipitación ya descargada. SPI estándar (McKee et al. 1993, ajuste gamma) y ET0 por Hargreaves–Samani (1985), igual que `lima_cloud/spi.py` y `lima_cloud/et0.py`.

In [ ]:
import math


def _monthly_totals(daily_precip_mm, days_per_month=30):
    n_full = len(daily_precip_mm) // days_per_month
    trimmed = daily_precip_mm[: n_full * days_per_month]
    return trimmed.reshape(n_full, days_per_month).sum(axis=1)


def rolling_sum(monthly_totals, scale_months):
    if len(monthly_totals) < scale_months:
        return np.array([])
    kernel = np.ones(scale_months)
    return np.convolve(monthly_totals, kernel, mode="valid")


def spi_from_series(daily_precip_mm, scale_months, days_per_month=30):
    monthly = _monthly_totals(daily_precip_mm, days_per_month)
    accum = rolling_sum(monthly, scale_months)
    if len(accum) < 8:
        return float("nan")
    current = accum[-1]
    zeros, nonzero = accum[accum <= 0], accum[accum > 0]
    q_zero = len(zeros) / len(accum)
    if len(nonzero) < 4:
        return float("nan")
    shape, loc, scale = stats.gamma.fit(nonzero, floc=0)
    cdf = q_zero if current <= 0 else q_zero + (1 - q_zero) * stats.gamma.cdf(current, shape, loc=loc, scale=scale)
    cdf = min(max(cdf, 1e-6), 1 - 1e-6)
    return float(stats.norm.ppf(cdf))


def count_consecutive_dry_days(daily_precip_mm, dry_threshold_mm=1.0):
    count = 0
    for v in daily_precip_mm[::-1]:
        if v < dry_threshold_mm:
            count += 1
        else:
            break
    return count


def extraterrestrial_radiation_mm_day(latitude_deg, day_of_year):
    lat_rad = math.radians(latitude_deg)
    dr = 1 + 0.033 * math.cos(2 * math.pi * day_of_year / 365)
    decl = 0.409 * math.sin(2 * math.pi * day_of_year / 365 - 1.39)
    xarg = min(max(-math.tan(lat_rad) * math.tan(decl), -1.0), 1.0)
    ws = math.acos(xarg)
    gsc = 0.0820
    ra_mj = (24 * 60 / math.pi) * gsc * dr * (ws * math.sin(lat_rad) * math.sin(decl) + math.cos(lat_rad) * math.cos(decl) * math.sin(ws))
    return ra_mj * 0.408


def et0_hargreaves_mm_day(t_mean_c, t_max_c, t_min_c, latitude_deg, day_of_year):
    ra = extraterrestrial_radiation_mm_day(latitude_deg, day_of_year)
    delta_t = max(t_max_c - t_min_c, 0.0)
    return 0.0023 * (t_mean_c + 17.8) * math.sqrt(delta_t) * ra


def classify_cur_class(vci):
    edges = VCI_CLASS_THRESHOLDS
    if np.isnan(vci):
        return 0
    if vci <= edges[0]:
        return 4
    if vci <= edges[1]:
        return 3
    if vci <= edges[2]:
        return 2
    if vci <= edges[3]:
        return 1
    return 0


def compute_dsi_score(spi3, vci, nddi):
    parts, weights = [], []
    if not np.isnan(spi3):
        parts.append(max(-3.0, min(3.0, -spi3)) / 3.0); weights.append(0.4)
    if not np.isnan(vci):
        parts.append((100.0 - max(0.0, min(100.0, vci))) / 100.0); weights.append(0.35)
    if not np.isnan(nddi):
        parts.append(max(0.0, min(1.0, nddi))); weights.append(0.25)
    if not parts:
        return float("nan")
    return float(np.average(parts, weights=weights))


print("Funciones SPI/ET0 listas.")

## 6. Ejecutar la adquisición real para el cuadrante de Mallacayán

Esto consulta Earth Engine en vivo — tarda entre ~30 s y unos minutos según la carga de los servidores de GEE (el histórico de 15 años de CHIRPS para el SPI es lo más lento).

In [ ]:
geom = quadrant_geometry(quad)
end_date = ee.Date(dt.datetime.utcnow().strftime("%Y-%m-%d"))

print("Consultando Sentinel-2 (NDVI/NDWI)...")
ndvi, ndwi = ndvi_ndwi_now(geom, end_date, S2_LOOKBACK_DAYS)
nddi = (ndvi - ndwi) / (ndvi + ndwi) if not np.isnan(ndvi + ndwi) and (ndvi + ndwi) != 0 else float("nan")

print("Consultando MODIS (LST)...")
lst = lst_celsius(geom, end_date, LST_LOOKBACK_DAYS)

print("Calculando climatología NDVI para VCI (6 años)...")
ndvi_min, ndvi_max = ndvi_historical_minmax(geom, end_date, VCI_HISTORY_YEARS)
vci = (ndvi - ndvi_min) / (ndvi_max - ndvi_min) * 100.0 if not np.isnan(ndvi_max - ndvi_min) and (ndvi_max - ndvi_min) != 0 else float("nan")

print("Consultando CHIRPS (15 años, para SPI/CDD)...")
precip = precip_daily_series(geom, end_date, max(SPI_HISTORY_YEARS * 365, CDD_WINDOW_DAYS))
spi1 = spi_from_series(precip, 1) if len(precip) else float("nan")
spi3 = spi_from_series(precip, 3) if len(precip) else float("nan")
spi6 = spi_from_series(precip, 6) if len(precip) else float("nan")
cdd = count_consecutive_dry_days(precip[-CDD_WINDOW_DAYS:], CDD_DRY_THRESHOLD_MM) if len(precip) else 0

print("Consultando ERA5-Land (temperatura, para ET0)...")
tmean, tmax, tmin = era5_temps(geom, end_date)
lat_center = (quad["lat_min"] + quad["lat_max"]) / 2.0
doy = dt.datetime.utcnow().timetuple().tm_yday
et0 = et0_hargreaves_mm_day(tmean, tmax, tmin, lat_center, doy) if not np.isnan(tmean) else float("nan")

print("Calculando tendencia NDVI a 14 días...")
ndvi_series_14 = ndvi_series(geom, end_date, TREND_WINDOW_DAYS)
ndvi_t = linear_trend(ndvi_series_14)
nddi_t = linear_trend(ndvi_series_14) * -1.0 if len(ndvi_series_14) else float("nan")
vci_t = float("nan")   # requiere serie VCI diaria — no disponible con datos puntuales (ver README de lima_cloud)
lst_t = float("nan")   # MOD11A2 es compuesto de 8 días — ventana de 14 días da como máximo 2 puntos

dsi_score = compute_dsi_score(spi3, vci, nddi)
cur_class = classify_cur_class(vci)

satelital = dict(ndvi=ndvi, ndwi=ndwi, nddi=nddi, lst=lst, vci=vci, dsi_score=dsi_score, cur_class=cur_class,
                  spi1=spi1, spi3=spi3, spi6=spi6, cdd=cdd, et0=et0,
                  nddi_t=nddi_t, ndvi_t=ndvi_t, vci_t=vci_t, lst_t=lst_t)
print("\nVariables satelitales/climáticas (cuadrante", NODE_QUADRANT_ID, "):")
for k, v in satelital.items():
    print(f"  {k:10s} = {v}")

## 7. Datos del sensor local (BME280) — entrada manual

Colab no tiene forma de leer el sensor físico instalado en Mallacayán. Ingresá acá la **última lectura real** que tengas del nodo (por ejemplo, del log de la MicroSD o del resumen diario que ya armamos en el paper). Estos valores **no entran como feature cruda del modelo** — el firmware real tampoco lo hace (ver `feature_vector.cpp`): se usan solo como chequeo de plausibilidad frente al dato satelital, igual que `sensor_validator.cpp` en el nodo.

In [ ]:
#@title Última lectura real del sensor local (editar y correr)

sensor_temp_c = 9.0      #@param {type:"number"}
sensor_hum_pct = 55.0    #@param {type:"number"}
sensor_pres_hpa = 665.3  #@param {type:"number"}
sensor_fecha = "2026-08-13"  #@param {type:"string"}

print(f"Sensor local ({sensor_fecha}): T={sensor_temp_c} C, HR={sensor_hum_pct} %, P={sensor_pres_hpa} hPa")

# Validación de plausibilidad frente al dato satelital (ERA5-Land), igual
# criterio que la Sección 4.5 del paper: diferencia > 6 °C se marca como
# discrepancia a revisar, no como error automático (el sesgo local vs ERA5
# medido en el paper fue de +3.24 °C, sistemático, no un error de sensor).
if not np.isnan(tmean):
    diff = sensor_temp_c - tmean
    estado = "OK (dentro de la banda esperada)" if abs(diff) <= 6.0 else "AVISO: revisar sensor o dato satelital"
    print(f"Diferencia vs. T media ERA5-Land ({tmean:.2f} C): {diff:+.2f} C -> {estado}")
else:
    print("No hay dato ERA5-Land disponible hoy para comparar (ver latencia de publicación, Sección 3.1 del paper).")

## 8. Construcción del vector de entrada x[0..19]

Orden **exacto** del contrato del firmware (`feature_vector.cpp`, líneas 104–128): 16 variables satelitales, fila/columna del cuadrante, y mes codificado como seno/coseno (nunca hardcodeado).

In [ ]:
month = int(sensor_fecha.split("-")[1])

x = [0.0] * 20
x[0] = satelital["ndvi"]
x[1] = satelital["ndwi"]
x[2] = satelital["nddi"]
x[3] = satelital["lst"]
x[4] = satelital["vci"]
x[5] = satelital["dsi_score"]
x[6] = float(satelital["cur_class"])
x[7] = satelital["spi1"]
x[8] = satelital["spi3"]
x[9] = satelital["spi6"]
x[10] = float(satelital["cdd"])
x[11] = satelital["et0"]
x[12] = satelital["nddi_t"]
x[13] = satelital["ndvi_t"]
x[14] = satelital["vci_t"]
x[15] = satelital["lst_t"]
x[16] = float(NODE_QUADRANT_ROW)
x[17] = float(NODE_QUADRANT_COL)
x[18] = math.sin(2 * math.pi * month / 12.0)
x[19] = math.cos(2 * math.pi * month / 12.0)

if any(np.isnan(v) for v in x):
    print("AVISO: hay NaN en el vector (variable sin dato disponible hoy) — el modelo real en el ESP32-S3")
    print("tampoco tiene forma de imputarlos; en producción esa lectura se descartaría hasta el próximo ciclo.")

print("x =", [round(v, 4) if not np.isnan(v) else float("nan") for v in x])

## 9. Modelo embebido — Random Forest (port literal del firmware compilado)

`votesForOnset()` de abajo es una traducción **si/entonces por si/entonces** de `drought_onset_model.h` (comentario del propio archivo: *"Onset de sequía 14d — RF óptimo bajo presupuesto 256KB (n_est=40, depth=4, min_leaf=30). GridSearch con TimeSeriesSplit (scoring=average_precision). NO editar a mano."*) — no es un modelo reentrenado ni una aproximación con scikit-learn. Los 40 árboles y sus umbrales exactos vienen del archivo tal cual está compilado en el nodo.

In [ ]:
def votes_for_onset(x):
    votes=[0,0]
    if x[1] <= -0.002109951921738684:
        if x[6] <= 0.5:
            if x[8] <= -1.4511500000953674:
                if x[14] <= 19.027764320373535:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 37.83342742919922:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[10] <= 20.5:
                if x[0] <= 0.24178653955459595:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[19] <= -0.6830126941204071:
                    votes[1]+=1
                else:
                    votes[1]+=1
    else:
        if x[1] <= 0.025233694352209568:
            if x[18] <= 0.25000000000000006:
                if x[3] <= 30.311549186706543:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[11] <= 3.6876879930496216:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[4] <= 28.50129795074463:
                if x[12] <= 0.017557775601744652:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[12] <= 0.9478624165058136:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[6] <= 0.5:
        if x[8] <= -1.4511500000953674:
            if x[4] <= 38.608421325683594:
                if x[13] <= 0.05498952604830265:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[2] <= 1.5896328687667847:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[0] <= 0.25443902611732483:
                if x[16] <= 1.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[9] <= -1.6718999743461609:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[9] <= -1.4374499917030334:
            if x[10] <= 54.5:
                if x[2] <= -1.6826788783073425:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[17] <= 0.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[0] <= 0.2389787808060646:
                if x[8] <= -0.9948999881744385:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[14] <= 7.229845285415649:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[8] <= -0.9948999881744385:
        if x[4] <= 37.70514678955078:
            if x[6] <= 0.5:
                if x[3] <= 19.354978561401367:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= -0.1460307613015175:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[7] <= -2.6158000230789185:
                if x[10] <= 20.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[15] <= 1.077254295349121:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[6] <= 0.5:
            if x[0] <= 0.26886264979839325:
                if x[19] <= -0.9330126941204071:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[2] <= 1.2035169005393982:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[9] <= -1.5703499913215637:
                if x[0] <= 0.14553788304328918:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= -0.16049452126026154:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[8] <= -0.9948999881744385:
        if x[3] <= 26.64146137237549:
            if x[7] <= -0.11980000138282776:
                if x[1] <= -0.031101166270673275:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[14] <= -4.852380990982056:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[14] <= -10.691983222961426:
                if x[9] <= -0.5460499823093414:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[6] <= 0.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
    else:
        if x[1] <= -0.028543257154524326:
            if x[6] <= 0.5:
                if x[13] <= -0.02068976778537035:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[12] <= -0.002830874640494585:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[5] <= 1.5:
                if x[4] <= 38.00168037414551:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[8] <= -0.7562000155448914:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[3] <= 23.702218055725098:
        if x[6] <= 0.5:
            if x[1] <= -0.020194614306092262:
                if x[4] <= 52.48038864135742:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[14] <= -18.807558059692383:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[12] <= -0.012808762025088072:
                if x[9] <= -1.6718999743461609:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 47.24648666381836:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[0] <= 0.2746490389108658:
            if x[5] <= 1.5:
                if x[7] <= -0.9038999974727631:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 38.79902648925781:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[4] <= 49.99357032775879:
                if x[1] <= 0.003117289044894278:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[15] <= -1.9697094559669495:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[0] <= 0.27446068823337555:
        if x[4] <= 38.79623603820801:
            if x[7] <= -0.9038999974727631:
                if x[1] <= -0.1449776142835617:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[5] <= 1.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[9] <= -1.4374499917030334:
                if x[4] <= 52.52038764953613:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[1] <= -0.002860885113477707:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[2] <= 1.5201126337051392:
            if x[9] <= -2.018649935722351:
                if x[15] <= 0.9256840944290161:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[2] <= 1.2338561415672302:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[11] <= 3.8690961599349976:
                if x[8] <= 0.696800023317337:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[10] <= 21.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
    if x[5] <= 1.5:
        if x[8] <= -1.4511500000953674:
            if x[9] <= 0.4294000118970871:
                if x[0] <= 0.2743788808584213:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[3] <= 31.09884738922119:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[18] <= -0.9330126941204071:
                if x[1] <= -0.0890595018863678:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[0] <= 0.26386958360671997:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[0] <= 0.2646523118019104:
            if x[10] <= 21.5:
                if x[11] <= 3.6876879930496216:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[4] <= 42.100921630859375:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[3] <= 27.088146209716797:
                if x[8] <= -0.7379499971866608:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 50.58766555786133:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[0] <= 0.2745263874530792:
        if x[4] <= 38.359283447265625:
            if x[18] <= 0.6830126941204071:
                if x[7] <= -0.8686999976634979:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[17] <= 3.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[10] <= 14.5:
                if x[2] <= 2.0939525365829468:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[0] <= 0.20562071353197098:
                    votes[0]+=1
                else:
                    votes[1]+=1
    else:
        if x[8] <= -1.542449951171875:
            if x[11] <= 3.8690961599349976:
                if x[13] <= 0.007182077970355749:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[2] <= 1.7095661163330078:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[8] <= -0.11205000057816505:
                if x[0] <= 0.34769895672798157:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[11] <= 3.2895349264144897:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[3] <= 23.701830863952637:
        if x[9] <= -1.4374499917030334:
            if x[8] <= -2.2856500148773193:
                if x[5] <= 1.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[5] <= 1.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[4] <= 39.58933448791504:
                if x[1] <= 0.008481056429445744:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[0] <= 0.27884557843208313:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[18] <= 0.25000000000000006:
            if x[5] <= 1.5:
                if x[18] <= -0.2500000000000001:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[0] <= 0.24364131689071655:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[11] <= 3.306349277496338:
                if x[3] <= 25.128382682800293:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[15] <= -1.4630724787712097:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[4] <= 50.0501651763916:
        if x[4] <= 37.70514678955078:
            if x[5] <= 1.5:
                if x[3] <= 15.494070529937744:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[8] <= -1.4511500000953674:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[7] <= -2.6158000230789185:
                if x[10] <= 20.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[6] <= 0.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
    else:
        if x[9] <= -1.5703499913215637:
            if x[10] <= 21.5:
                if x[8] <= -1.3825999796390533:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[13] <= 0.004114137496799231:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[11] <= 3.3088799715042114:
                if x[16] <= 4.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[17] <= 4.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[6] <= 0.5:
        if x[8] <= -1.4511500000953674:
            if x[1] <= 0.007409064332023263:
                if x[19] <= -0.6830126941204071:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[18] <= -0.6830126941204071:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[1] <= 0.0022264906438067555:
                if x[14] <= -0.9022957682609558:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[19] <= -0.9330126941204071:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[11] <= 3.6876879930496216:
            if x[9] <= -0.9700499773025513:
                if x[2] <= 5.098600625991821:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[16] <= 2.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[7] <= -0.4877000004053116:
                if x[16] <= 4.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[12] <= -0.001724941423162818:
                    votes[0]+=1
                else:
                    votes[1]+=1
    if x[10] <= 21.5:
        if x[6] <= 0.5:
            if x[11] <= 4.007659435272217:
                if x[2] <= 1.5995318293571472:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[14] <= 1.9651040434837341:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[8] <= -0.7010000050067902:
                if x[15] <= -8.275016784667969:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= -0.16016839444637299:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[9] <= -0.5460499823093414:
            if x[19] <= -0.6830126941204071:
                if x[16] <= 1.5:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[1] <= -0.0046502454206347466:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[0] <= 0.23865963518619537:
                if x[5] <= 0.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= 0.02879216056317091:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[6] <= 0.5:
        if x[8] <= -1.4511500000953674:
            if x[9] <= 0.4294000118970871:
                if x[14] <= 23.209747314453125:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[3] <= 31.09884738922119:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[9] <= -1.6718999743461609:
                if x[14] <= 5.458775043487549:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[10] <= 78.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
    else:
        if x[0] <= 0.2646859288215637:
            if x[8] <= -0.7336499989032745:
                if x[7] <= -0.45454999804496765:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[12] <= 0.24003320932388306:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[9] <= -1.503849983215332:
                if x[4] <= 49.87775993347168:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 46.80460548400879:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[0] <= 0.2746075838804245:
        if x[6] <= 0.5:
            if x[10] <= 10.5:
                if x[4] <= 36.34266662597656:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[1] <= -0.035184163600206375:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[7] <= -0.9038999974727631:
                if x[1] <= -0.14614979177713394:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= -0.011610358487814665:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[11] <= 3.8690961599349976:
            if x[16] <= 4.5:
                if x[19] <= 0.6830126941204071:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[9] <= -0.16875000298023224:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[1] <= 0.002984339836984873:
                if x[12] <= 0.06816298887133598:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[11] <= 4.01864218711853:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[11] <= 3.806143045425415:
        if x[0] <= 0.26366257667541504:
            if x[5] <= 1.5:
                if x[16] <= 1.5:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[1] <= 0.0017689052619971335:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[9] <= -1.5703499913215637:
                votes[1]+=1
            else:
                if x[2] <= 1.1593675017356873:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[1] <= 0.0010778920259326696:
            if x[13] <= 0.007968185469508171:
                if x[8] <= -1.4511500000953674:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[8] <= -1.2461999654769897:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[4] <= 37.17397117614746:
                if x[8] <= -1.7394999265670776:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[12] <= -8.877309322357178:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[6] <= 0.5:
        if x[4] <= 50.375091552734375:
            if x[10] <= 11.5:
                if x[1] <= -0.010503090918064117:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[1] <= 0.021239906549453735:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[2] <= 1.5548232197761536:
                if x[7] <= -2.6158000230789185:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[11] <= 3.3088799715042114:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[0] <= 0.26354601979255676:
            if x[7] <= -0.4877000004053116:
                if x[4] <= 40.83010482788086:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[12] <= -0.001514521543867886:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[14] <= 7.19672155380249:
                if x[9] <= -1.6718999743461609:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 49.58370399475098:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[6] <= 0.5:
        if x[10] <= 21.5:
            if x[4] <= 38.84354019165039:
                if x[13] <= -0.019302104599773884:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 54.80378341674805:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[4] <= 50.66910171508789:
                if x[11] <= 3.6711539030075073:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[15] <= -2.4285991191864014:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[0] <= 0.2646523118019104:
            if x[8] <= -0.7336499989032745:
                if x[14] <= -12.127185821533203:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[12] <= 0.24180206656455994:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[11] <= 4.1308205127716064:
                if x[7] <= 0.11639999970793724:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[14] <= 7.19672155380249:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[14] <= 5.926922798156738:
        if x[11] <= 3.6876879930496216:
            if x[0] <= 0.24838344752788544:
                if x[19] <= -0.6830126941204071:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[0] <= 0.27710869908332825:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[5] <= 1.5:
                if x[14] <= -0.044458383694291115:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[7] <= -0.8583000004291534:
                    votes[1]+=1
                else:
                    votes[1]+=1
    else:
        if x[4] <= 47.313493728637695:
            if x[9] <= -1.6718999743461609:
                if x[5] <= 0.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[10] <= 49.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[10] <= 20.5:
                if x[9] <= -1.5703499913215637:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[9] <= -1.599799931049347:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[8] <= -0.9948999881744385:
        if x[3] <= 25.50452995300293:
            if x[15] <= -5.323265075683594:
                if x[9] <= -1.419600009918213:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[3] <= 23.334228515625:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[6] <= 0.5:
                if x[4] <= 36.6901798248291:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= -0.14679493755102158:
                    votes[1]+=1
                else:
                    votes[1]+=1
    else:
        if x[6] <= 0.5:
            if x[2] <= 1.2849662899971008:
                if x[4] <= 42.3844051361084:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[11] <= 3.3088799715042114:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[19] <= 0.6830126941204071:
                if x[8] <= 0.16279999911785126:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[9] <= -1.5703499913215637:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[4] <= 39.57249450683594:
        if x[18] <= 0.6830126941204071:
            if x[7] <= -0.8686999976634979:
                if x[4] <= 36.86797904968262:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[5] <= 1.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[12] <= 0.2622143030166626:
                votes[0]+=1
            else:
                if x[14] <= -7.435013055801392:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[1] <= -0.013495900202542543:
            if x[9] <= -1.4374499917030334:
                if x[4] <= 51.95855712890625:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 46.39856719970703:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[16] <= 4.5:
                if x[4] <= 63.21228790283203:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[19] <= -0.9330126941204071:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[5] <= 1.5:
        if x[8] <= -1.4511500000953674:
            if x[13] <= -0.047290390357375145:
                if x[5] <= 0.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= 0.0069449106231331825:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[12] <= -5.875386714935303:
                if x[15] <= 8.872822761535645:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[14] <= -13.026733875274658:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[0] <= 0.2646523118019104:
            if x[7] <= -0.9038999974727631:
                if x[9] <= 0.489300012588501:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[7] <= -0.5932500064373016:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[12] <= 0.6224040985107422:
                if x[7] <= -2.6158000230789185:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[9] <= -0.7348500192165375:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[11] <= 3.806143045425415:
        if x[8] <= -0.7562000155448914:
            if x[6] <= 0.5:
                if x[11] <= 3.4809354543685913:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[10] <= 58.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[19] <= -0.9330126941204071:
                if x[0] <= 0.24459733814001083:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[2] <= 1.804413616657257:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[6] <= 0.5:
            if x[8] <= -1.4511500000953674:
                if x[14] <= 18.558894157409668:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[19] <= 0.24999999999999992:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[4] <= 38.90551567077637:
                if x[8] <= -1.4511500000953674:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[12] <= 0.24195606261491776:
                    votes[0]+=1
                else:
                    votes[1]+=1
    if x[8] <= -0.9948999881744385:
        if x[5] <= 1.5:
            if x[1] <= 0.013137095607817173:
                if x[4] <= 36.35371017456055:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[4] <= 38.07658386230469:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[7] <= -0.45454999804496765:
                if x[9] <= 0.4294000118970871:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[0] <= 0.215731643140316:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[5] <= 1.5:
            if x[1] <= 0.003986341180279851:
                if x[0] <= 0.26637089252471924:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[11] <= 3.3088799715042114:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[0] <= 0.26941289007663727:
                if x[2] <= -8.764519214630127:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[18] <= -0.6830126941204071:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[5] <= 1.5:
        if x[1] <= 0.0050625556614249945:
            if x[7] <= -0.9038999974727631:
                if x[4] <= 52.78886413574219:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[15] <= -0.0014961957931518555:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[0] <= 0.2725844234228134:
                if x[7] <= -1.3153499960899353:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[16] <= 4.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[18] <= 0.6830126941204071:
            if x[0] <= 0.23893731087446213:
                if x[19] <= 0.24999999999999992:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[17] <= 2.5:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[16] <= 0.5:
                if x[9] <= -0.6566500067710876:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[15] <= 10.798975467681885:
                    votes[0]+=1
                else:
                    votes[1]+=1
    if x[0] <= 0.2746075838804245:
        if x[5] <= 1.5:
            if x[4] <= 38.60887145996094:
                if x[8] <= -1.4511500000953674:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[7] <= -2.000200033187866:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[10] <= 20.5:
                if x[8] <= -0.7010000050067902:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[0] <= 0.24079453945159912:
                    votes[1]+=1
                else:
                    votes[1]+=1
    else:
        if x[2] <= 1.598714292049408:
            if x[1] <= -0.06312720477581024:
                votes[1]+=1
            else:
                if x[16] <= 4.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[13] <= 0.033935584127902985:
                if x[8] <= -1.542449951171875:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[10] <= 17.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[10] <= 18.5:
        if x[5] <= 1.5:
            if x[8] <= -0.9948999881744385:
                if x[11] <= 4.007659435272217:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[8] <= -0.7010000050067902:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[9] <= -0.4892999976873398:
                if x[13] <= 0.015894107520580292:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[4] <= 60.46682357788086:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[1] <= 0.0038106540450826287:
            if x[8] <= -1.2452999949455261:
                if x[19] <= -0.6830126941204071:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[9] <= -0.04475000128149986:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[3] <= 30.048917770385742:
                if x[17] <= 1.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                votes[1]+=1
    if x[11] <= 3.806143045425415:
        if x[0] <= 0.2724364399909973:
            if x[9] <= -0.6863000094890594:
                if x[1] <= 0.008999607991427183:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[18] <= 0.25000000000000006:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[4] <= 39.357242584228516:
                votes[1]+=1
            else:
                if x[16] <= 4.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[4] <= 36.96962928771973:
            if x[3] <= 12.193062782287598:
                if x[19] <= 0.24999999999999992:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[3] <= 25.30337619781494:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[5] <= 0.5:
                if x[7] <= -1.8531000018119812:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[3] <= 28.533668518066406:
                    votes[0]+=1
                else:
                    votes[1]+=1
    if x[0] <= 0.2733222097158432:
        if x[5] <= 1.5:
            if x[9] <= -1.9047999382019043:
                if x[12] <= 14.96291446685791:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 38.83970260620117:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[18] <= 0.6830126941204071:
                if x[10] <= 20.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[9] <= -0.6566500067710876:
                    votes[0]+=1
                else:
                    votes[1]+=1
    else:
        if x[11] <= 4.01864218711853:
            if x[16] <= 4.5:
                if x[8] <= -0.7562000155448914:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[14] <= -0.06556422589346766:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[15] <= -2.058199405670166:
                if x[9] <= -0.7264499962329865:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[15] <= -0.43509846925735474:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[5] <= 1.5:
        if x[1] <= 0.005200369982048869:
            if x[10] <= 21.5:
                if x[18] <= -0.9330126941204071:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[7] <= -2.000200033187866:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[12] <= 0.4992251545190811:
                if x[18] <= -0.6830126941204071:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 37.2995719909668:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[3] <= 24.91032886505127:
            if x[4] <= 47.19503593444824:
                if x[18] <= 0.6830126941204071:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[15] <= -4.0933005809783936:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[14] <= 14.203740119934082:
                if x[0] <= 0.24128882586956024:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[10] <= 66.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
    if x[7] <= -0.9038999974727631:
        if x[0] <= 0.2746094614267349:
            if x[19] <= -0.6830126941204071:
                if x[8] <= 0.23829999193549156:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 36.87550354003906:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[9] <= -1.599799931049347:
                if x[10] <= 20.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[12] <= 0.5697693228721619:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[11] <= 3.6876879930496216:
            if x[2] <= 1.5438445210456848:
                if x[14] <= -34.88887023925781:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[7] <= 0.8307500183582306:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[6] <= 0.5:
                if x[8] <= -1.9054999351501465:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[9] <= -0.7264499962329865:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[9] <= -0.7348500192165375:
        if x[4] <= 52.78803634643555:
            if x[7] <= -0.4877000004053116:
                if x[4] <= 37.06197929382324:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= -0.07196726649999619:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[9] <= -2.018649935722351:
                if x[3] <= 26.268190383911133:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 53.316293716430664:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[10] <= 40.5:
            if x[6] <= 0.5:
                if x[18] <= -0.9330126941204071:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[8] <= -0.9948999881744385:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[3] <= 28.407257080078125:
                if x[2] <= 2.733590006828308:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[14] <= -12.194112300872803:
                    votes[1]+=1
                else:
                    votes[1]+=1
    if x[1] <= -0.0025628675939515233:
        if x[8] <= -1.2461999654769897:
            if x[16] <= 4.5:
                if x[5] <= 1.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[1] <= -0.05437726899981499:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[8] <= -0.08265000209212303:
                if x[6] <= 0.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[14] <= -8.750662326812744:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[1] <= 0.025877193547785282:
            if x[8] <= -2.010249972343445:
                if x[16] <= 2.5:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[15] <= -5.185457944869995:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[2] <= 1.1372274160385132:
                if x[6] <= 0.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[0] <= 0.2551029324531555:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[8] <= -0.9948999881744385:
        if x[4] <= 37.19650459289551:
            if x[10] <= 11.5:
                if x[8] <= -1.4511500000953674:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[16] <= 4.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[11] <= 4.150537729263306:
                if x[4] <= 45.52791786193848:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[2] <= 1.5826770067214966:
                    votes[1]+=1
                else:
                    votes[1]+=1
    else:
        if x[1] <= -0.0007494656019844115:
            if x[6] <= 0.5:
                if x[11] <= 3.3088799715042114:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 46.29710388183594:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[0] <= 0.34294120967388153:
                if x[2] <= 1.1371976733207703:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[14] <= -17.465834617614746:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[11] <= 3.806143045425415:
        if x[5] <= 1.5:
            if x[4] <= 39.89326286315918:
                if x[8] <= -1.3977499902248383:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[2] <= 1.3007922172546387:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[0] <= 0.2771218568086624:
                if x[18] <= 0.6830126941204071:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[2] <= 2.325320601463318:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[4] <= 36.438629150390625:
            if x[11] <= 4.1308205127716064:
                if x[10] <= 88.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[7] <= -1.1226499676704407:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[8] <= -1.4511500000953674:
                if x[1] <= -0.003127820324152708:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[3] <= 27.826319694519043:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[6] <= 0.5:
        if x[8] <= -1.4511500000953674:
            if x[1] <= -0.005772766890004277:
                if x[4] <= 36.9869441986084:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[4] <= 39.49666213989258:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[0] <= 0.2448430061340332:
                if x[10] <= 14.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[13] <= -0.013916201423853636:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[0] <= 0.24541117995977402:
            if x[14] <= -42.53268814086914:
                if x[14] <= -59.7058162689209:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[8] <= -1.4511500000953674:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[13] <= 0.021479162387549877:
                if x[4] <= 53.093610763549805:
                    votes[1]+=1
                else:
                    votes[0]+=1
            else:
                if x[15] <= -2.058199405670166:
                    votes[1]+=1
                else:
                    votes[0]+=1
    if x[1] <= -0.0024133564438670874:
        if x[7] <= -0.9038999974727631:
            if x[5] <= 1.5:
                if x[19] <= -0.6830126941204071:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[15] <= -1.0626392364501953:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[8] <= -0.7562000155448914:
                if x[7] <= -0.6457499861717224:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[12] <= 0.2294401079416275:
                    votes[0]+=1
                else:
                    votes[0]+=1
    else:
        if x[18] <= -0.6830126941204071:
            if x[3] <= 26.864315032958984:
                if x[13] <= -0.025344280526041985:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[16] <= 2.5:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[4] <= 36.3143367767334:
                if x[13] <= -0.11794847622513771:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[8] <= -0.08265000209212303:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[8] <= -0.9948999881744385:
        if x[6] <= 0.5:
            if x[4] <= 38.33960151672363:
                if x[10] <= 9.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[7] <= -2.000200033187866:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[4] <= 39.22529411315918:
                if x[7] <= -0.45454999804496765:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[14] <= -9.88499927520752:
                    votes[1]+=1
                else:
                    votes[0]+=1
    else:
        if x[1] <= -0.009419510141015053:
            if x[8] <= -0.7700999975204468:
                if x[2] <= -1.527442455291748:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[5] <= 1.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
        else:
            if x[12] <= 0.559532105922699:
                if x[9] <= -0.7116000056266785:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[13] <= 0.005877539049834013:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[1] <= -0.008153030648827553:
        if x[8] <= -1.2461999654769897:
            if x[4] <= 37.30580139160156:
                if x[6] <= 0.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[7] <= -2.6158000230789185:
                    votes[1]+=1
                else:
                    votes[1]+=1
        else:
            if x[5] <= 1.5:
                if x[13] <= -0.0029651887016370893:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[12] <= -0.010039696004241705:
                    votes[0]+=1
                else:
                    votes[1]+=1
    else:
        if x[18] <= -0.6830126941204071:
            if x[3] <= 26.838873863220215:
                if x[11] <= 4.1308205127716064:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[4] <= 35.35114669799805:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[0] <= 0.27651844918727875:
                if x[16] <= 1.5:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[12] <= 0.5648268163204193:
                    votes[0]+=1
                else:
                    votes[0]+=1
    if x[4] <= 40.10766410827637:
        if x[7] <= -0.9038999974727631:
            if x[16] <= 4.5:
                if x[4] <= 36.71581840515137:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[19] <= 0.9330126941204071:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[5] <= 1.5:
                if x[10] <= 79.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[13] <= -0.0777728222310543:
                    votes[0]+=1
                else:
                    votes[1]+=1
    else:
        if x[3] <= 24.58669090270996:
            if x[11] <= 4.150537729263306:
                if x[8] <= -0.7336499989032745:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[15] <= -3.3413236141204834:
                    votes[1]+=1
                else:
                    votes[0]+=1
        else:
            if x[19] <= -0.6830126941204071:
                if x[8] <= -0.2487500011920929:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[10] <= 20.5:
                    votes[0]+=1
                else:
                    votes[1]+=1
    if x[4] <= 51.01192855834961:
        if x[6] <= 0.5:
            if x[7] <= -0.9038999974727631:
                if x[19] <= -0.6830126941204071:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[8] <= -0.9948999881744385:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[9] <= -1.4374499917030334:
                if x[10] <= 21.5:
                    votes[1]+=1
                else:
                    votes[1]+=1
            else:
                if x[3] <= 23.4280366897583:
                    votes[0]+=1
                else:
                    votes[1]+=1
    else:
        if x[2] <= 1.5203546285629272:
            if x[12] <= -0.5246057212352753:
                if x[15] <= 4.666030406951904:
                    votes[0]+=1
                else:
                    votes[0]+=1
            else:
                if x[11] <= 3.3088799715042114:
                    votes[0]+=1
                else:
                    votes[0]+=1
        else:
            if x[14] <= 1.410898745059967:
                if x[11] <= 4.027376651763916:
                    votes[0]+=1
                else:
                    votes[1]+=1
            else:
                if x[8] <= -0.7336499989032745:
                    votes[0]+=1
                else:
                    votes[0]+=1
    return votes[1]

In [ ]:
def predict(x):
    return 1 if votes_for_onset(x) * 2 > 40 else 0


def alert_level(votes):
    if votes >= 31:
        return "ROJO"
    if votes >= 19:
        return "AMARILLO"
    return "VERDE"

## 10. Inferencia sobre el vector real (satélite + validación con sensor local)

In [ ]:
votes = votes_for_onset(x)
pred = predict(x)
nivel = alert_level(votes)

print(f"Cuadrante:        {NODE_QUADRANT_ID}")
print(f"Fecha:             {sensor_fecha}")
print(f"Votos (0-40):      {votes}")
print(f"predict() 0/1:     {pred}")
print(f"Nivel de alerta:   {nivel}  (umbrales: >=31 ROJO, >=19 AMARILLO, si no VERDE — igual que el Algoritmo 3 del paper)")

## Notas finales

- Si algún campo satelital salió `nan`, revisá la latencia de publicación de cada producto (Sección 3.1 del paper: MODIS ~17 días, CHIRPS ~45 días, ERA5-Land unos días a un par de semanas) — es esperable, no un bug.
- `dsi_score`/`cur_class` siguen siendo la heurística propia de `lima_cloud`, no la fórmula exacta de entrenamiento del equipo de ML — ver advertencia en la Sección 4.
- Este notebook no reentrena el modelo: usa el clasificador ya entrenado y compilado, portado literalmente desde `drought_onset_model.h`. Reentrenar con datos reales de la cuenca (en vez de los vectores sintéticos usados originalmente) sigue pendiente, tal como se documenta en el paper.